In [52]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [53]:
match_summary = pd.read_csv("../data/processed/match_summary.csv")

match_summary.head()

,match_id,date,season,venue,city,toss_winner,toss_decision,match_won_by,player_of_match,team1,team2,first_innings_score,first_innings_wickets,second_innings_score,second_innings_wickets,team1_win,bat_first_won,chasing_team_won
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangaluru,field,Kolkata Knight Riders,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangaluru,222,3,82,10,1,1,0
1,335983,2008-04-19,2007/08,Punjab Cricket Association Stadium,Chandigarh,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey,Chennai Super Kings,Punjab Kings,240,5,207,4,1,1,0
2,335984,2008-04-19,2007/08,Arun Jaitley Stadium,Delhi,Rajasthan Royals,bat,Delhi Capitals,MF Maharoof,Rajasthan Royals,Delhi Capitals,129,8,132,1,0,0,1
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,bat,Royal Challengers Bangaluru,MV Boucher,Mumbai Indians,Royal Challengers Bangaluru,165,6,166,5,0,0,1
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey,Deccan Chargers,Kolkata Knight Riders,110,10,112,5,0,0,1


In [54]:
match_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1187 entries, 0 to 1186
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   match_id                1187 non-null   int64 
 1   date                    1187 non-null   object
 2   season                  1187 non-null   object
 3   venue                   1187 non-null   object
 4   city                    1187 non-null   object
 5   toss_winner             1187 non-null   object
 6   toss_decision           1187 non-null   object
 7   match_won_by            1187 non-null   object
 8   player_of_match         1187 non-null   object
 9   team1                   1187 non-null   object
 10  team2                   1187 non-null   object
 11  first_innings_score     1187 non-null   int64 
 12  first_innings_wickets   1187 non-null   int64 
 13  second_innings_score    1187 non-null   int64 
 14  second_innings_wickets  1187 non-null   int64 
 15  team

In [55]:
match_summary["team1_win"] = (
    match_summary["match_won_by"] == match_summary["team1"]
).astype(int)

match_summary[["team1", "match_won_by", "team1_win"]].head()

,team1,match_won_by,team1_win
0,Kolkata Knight Riders,Kolkata Knight Riders,1
1,Chennai Super Kings,Chennai Super Kings,1
2,Rajasthan Royals,Delhi Capitals,0
3,Mumbai Indians,Royal Challengers Bangaluru,0
4,Deccan Chargers,Kolkata Knight Riders,0


In [56]:
features = match_summary[
    [
        "team1",
        "team2",
        "venue",
        "toss_winner",
        "toss_decision"
    ]
].copy()

target = match_summary["team1_win"]

In [57]:
encoders = {}

for column in features.columns:
    le = LabelEncoder()
    features[column] = le.fit_transform(features[column])
    encoders[column] = le

In [58]:
for col in encoders:
    print("\n", col)
    print(encoders[col].classes_[:10])


 team1
['Chennai Super Kings' 'Deccan Chargers' 'Delhi Capitals' 'Gujarat Lions'
 'Gujarat Titans' 'Kochi Tuskers Kerala' 'Kolkata Knight Riders'
 'Lucknow Super Giants' 'Mumbai Indians' 'Pune Warriors']

 team2
['Chennai Super Kings' 'Deccan Chargers' 'Delhi Capitals' 'Gujarat Lions'
 'Gujarat Titans' 'Kochi Tuskers Kerala' 'Kolkata Knight Riders'
 'Lucknow Super Giants' 'Mumbai Indians' 'Pune Warriors']

 venue
['Arun Jaitley Stadium' 'Barabati Stadium'
 'Barsapara Cricket Stadium, Guwahati'
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow'
 'Brabourne Stadium' 'Buffalo Park' 'De Beers Diamond Oval'
 'Dr DY Patil Sports Academy'
 'Dr YS Rajasekhara Reddy ACA-VDCA Cricket Stadium'
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam']

 toss_winner
['Chennai Super Kings' 'Deccan Chargers' 'Delhi Capitals' 'Gujarat Lions'
 'Gujarat Titans' 'Kochi Tuskers Kerala' 'Kolkata Knight Riders'
 'Lucknow Super Giants' 'Mumbai Indians' 'Pune Warriors']

 t

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

In [60]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [61]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 48.74%


In [62]:
joblib.dump(model, "../models/model.pkl")

joblib.dump(encoders, "../models/encoders.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [63]:
encoders = joblib.load("../models/encoders.pkl")

for col in encoders:
    print("\n", col)
    print(encoders[col].classes_[:10])


 team1
['Chennai Super Kings' 'Deccan Chargers' 'Delhi Capitals' 'Gujarat Lions'
 'Gujarat Titans' 'Kochi Tuskers Kerala' 'Kolkata Knight Riders'
 'Lucknow Super Giants' 'Mumbai Indians' 'Pune Warriors']

 team2
['Chennai Super Kings' 'Deccan Chargers' 'Delhi Capitals' 'Gujarat Lions'
 'Gujarat Titans' 'Kochi Tuskers Kerala' 'Kolkata Knight Riders'
 'Lucknow Super Giants' 'Mumbai Indians' 'Pune Warriors']

 venue
['Arun Jaitley Stadium' 'Barabati Stadium'
 'Barsapara Cricket Stadium, Guwahati'
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow'
 'Brabourne Stadium' 'Buffalo Park' 'De Beers Diamond Oval'
 'Dr DY Patil Sports Academy'
 'Dr YS Rajasekhara Reddy ACA-VDCA Cricket Stadium'
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam']

 toss_winner
['Chennai Super Kings' 'Deccan Chargers' 'Delhi Capitals' 'Gujarat Lions'
 'Gujarat Titans' 'Kochi Tuskers Kerala' 'Kolkata Knight Riders'
 'Lucknow Super Giants' 'Mumbai Indians' 'Pune Warriors']

 t